In [ ]:
# Temperature Scaling Deep Dive

## Goal

Understand how temperature modifies token probability distributions
and affects output diversity and stability.

We will compare:

- temperature = 0.3
- temperature = 0.7
- temperature = 1.0
- temperature = 1.3

All other variables remain constant.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

In [ ]:
prompt = """
Explain photosynthesis.

Return exactly 5 bullet points.
Each bullet must:
- Start with "- "
- Contain at most 12 words
- Use simple vocabulary
""".strip()

In [ ]:
def generate(prompt, temperature, seed=0):
    set_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
temperatures = [0.3, 0.7, 1.0, 1.3]

for temp in temperatures:
    print(f"\n=== Temperature = {temp} ===\n")
    print(generate(prompt, temperature=temp, seed=42))

In [ ]:
## Observations

Temperature 0.3:
- Output is highly stable
- Very low variation
- Strong adherence to format

Temperature 0.7:
- Slight wording differences
- Still stable and coherent

Temperature 1.0:
- Clear variation
- Minor format drift possible

Temperature 1.3:
- Larger variation
- Increased risk of format violations
- Occasional drift or redundancy

## Key Insight

Temperature does not change the model.
It reshapes the probability distribution.

Lower temperature sharpens the distribution.
Higher temperature flattens it.

Flattened distributions allow lower-probability tokens to be selected,
increasing diversity but also instability.